# Dialforge Cloud Benchmark

Runs the Dialforge AI stack on a **Google Colab T4 GPU** instead of your PC. It benchmarks **Qwen 3 1.7B / 4B / 8B**, **faster-whisper small.en**, **Chatterbox Nano**, and synthetic **STT → LLM → TTS** turns.

Choose **Runtime → Change runtime type → T4 GPU**, save it, then choose **Runtime → Run all**. No SIP credentials are needed and no real phone calls are made. All models stay on the temporary Colab machine.

> If setup fails, the notebook now prints the useful error directly instead of leaving the later cells red.

In [ ]:
# Benchmark controls.
QWEN_MODELS = ['qwen3:1.7b', 'qwen3:4b', 'qwen3:8b']
QWEN_REPEATS = 3
COMPONENT_REPEATS = 3
PIPELINE_TURNS = 5
PIPELINE_MODEL = 'qwen3:4b'
print('Models:', ', '.join(QWEN_MODELS))

In [ ]:
# Verify that Colab actually assigned an NVIDIA GPU.
import shutil, subprocess
if not shutil.which('nvidia-smi'):
    raise RuntimeError('No NVIDIA GPU is attached. Choose Runtime > Change runtime type > T4 GPU, then reconnect.')
subprocess.run(['nvidia-smi'], check=True)

In [ ]:
# Install Ollama. Colab has no normal systemd service, so Dialforge starts it manually later.
import os, shutil, subprocess
subprocess.run(['apt-get', 'update', '-qq'], check=True)
subprocess.run(['apt-get', 'install', '-y', '-qq', 'pciutils', 'curl', 'ca-certificates'], check=True)
install = subprocess.run(
    'curl -fsSL https://ollama.com/install.sh | sh',
    shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)
print(install.stdout[-5000:])
# The installer can warn that systemd is unavailable in Colab. That is fine if the CLI exists.
if not shutil.which('ollama'):
    raise RuntimeError(f'Ollama CLI was not installed. Installer exit code: {install.returncode}')
print('Ollama CLI:', shutil.which('ollama'))
subprocess.run(['ollama', '--version'], check=True)

In [ ]:
# Install the AI-side versions used by the Dialforge beta runtime.
import subprocess, sys
def pip_install(*args):
    cmd = [sys.executable, '-m', 'pip', 'install', '--no-cache-dir', *args]
    print('>', ' '.join(cmd))
    subprocess.run(cmd, check=True)
pip_install('--upgrade', 'pip', 'wheel', 'setuptools')
pip_install('torch==2.6.0', 'torchaudio==2.6.0', '--index-url', 'https://download.pytorch.org/whl/cu124')
pip_install('psutil==7.0.0', 'requests==2.32.5', 'numpy<2', 'faster-whisper==1.2.0')
pip_install('chatterbox-tts')
pip_install('--force-reinstall', '--no-deps', 'git+https://github.com/resemble-ai/Perth.git@ff1c8ac55a976971245cdd53c18d6131ca00d993')
pip_install('--force-reinstall', '--no-deps', 'git+https://github.com/resemble-ai/chatterbox.git@5de7a54aa4e5e2baadb0182dde554908b48b85c2')
# Verify the exact features Dialforge needs before the long benchmark starts.
import inspect, torch, torchaudio, perth
from chatterbox.tts_turbo import ChatterboxTurboTTS
from faster_whisper import WhisperModel
sig = inspect.signature(ChatterboxTurboTTS.from_pretrained)
if 'nano' not in sig.parameters:
    raise RuntimeError('Pinned Chatterbox build does not expose Nano support.')
print('PyTorch:', torch.__version__, '| CUDA:', torch.version.cuda, '| CUDA available:', torch.cuda.is_available())
print('Torchaudio:', torchaudio.__version__)
print('Chatterbox Nano: OK')
print('Perth watermark: OK')
print('faster-whisper: OK')

In [ ]:
# Start Ollama in the background and download the public benchmark runner.
import os, pathlib, subprocess, time, urllib.request, requests
def ollama_ready():
    try:
        return requests.get('http://127.0.0.1:11434/api/tags', timeout=2).ok
    except Exception:
        return False
OLLAMA_LOG_PATH = '/content/ollama.log'
if not ollama_ready():
    env = os.environ.copy()
    env['OLLAMA_HOST'] = '127.0.0.1:11434'
    env['OLLAMA_ORIGINS'] = '*'
    log = open(OLLAMA_LOG_PATH, 'w')
    ollama_proc = subprocess.Popen(['ollama', 'serve'], stdout=log, stderr=subprocess.STDOUT, env=env)
    for _ in range(60):
        if ollama_ready():
            break
        if ollama_proc.poll() is not None:
            break
        time.sleep(1)
if not ollama_ready():
    tail = pathlib.Path(OLLAMA_LOG_PATH).read_text(errors='replace')[-8000:] if pathlib.Path(OLLAMA_LOG_PATH).exists() else '(no log created)'
    print('----- /content/ollama.log -----')
    print(tail)
    raise RuntimeError('Ollama failed to start. The server log is printed above.')
runner = '/content/dialforge_colab_benchmark.py'
urllib.request.urlretrieve('https://raw.githubusercontent.com/SumamaAhmed69/Axemetric-Caller-Beta-Runtime/main/benchmarks/dialforge_colab_benchmark.py', runner)
print('Ollama ready.')
print('Benchmark runner:', runner)
subprocess.run(['ollama', 'ps'], check=False)

In [ ]:
# Run the benchmark. First run downloads Qwen, Whisper and Chatterbox models onto this temporary Colab VM.
import subprocess, sys
cmd = [sys.executable, '/content/dialforge_colab_benchmark.py',
       '--models', *QWEN_MODELS,
       '--qwen-repeats', str(QWEN_REPEATS),
       '--component-repeats', str(COMPONENT_REPEATS),
       '--pipeline-turns', str(PIPELINE_TURNS),
       '--pipeline-model', PIPELINE_MODEL,
       '--output-dir', '/content/dialforge-benchmark']
print('Starting Dialforge benchmark...')
subprocess.run(cmd, check=True)
print('Benchmark complete.')

In [ ]:
# Display the finished report inside Colab.
from IPython.display import HTML, display
report_html = '/content/dialforge-benchmark/dialforge-benchmark-report.html'
display(HTML(open(report_html, encoding='utf-8').read()))

In [ ]:
# Optional: download the two small result files to your PC.
# from google.colab import files
# files.download('/content/dialforge-benchmark/dialforge-benchmark-report.html')
# files.download('/content/dialforge-benchmark/dialforge-benchmark-report.json')